### Automated Search (Optuna) after Hard-Coded Analysis

This automated optimization is performed **after** the preliminary hard-coded parameter analysis conducted in  
`experiments/parameter-two-stage-finding.ipynb`, and is restricted to the development split.

The hard-coded exploration identified a stable performance peak for the two-stage strategy around the following configuration:

- **C** = 1.0  
- **min_rule_purity** = 0.95  
- **min_rule_support** = 40  
- **macro_f1** ≈ 0.721  
- **rule_coverage** ≈ 0.156  

This setting isolates approximately 15–20% of the samples through highly confident deterministic rules, while delegating the remaining ambiguous instances to the probabilistic classifier.

The automated search yields a slightly higher best score:

- **Best F1** ≈ 0.722  
- **Best parameters**:  
  - **C** ≈ 0.97  
  - **min_rule_purity** ≈ 0.93  
  - **min_rule_support** = 30  

The marginal improvement over the hard-coded configuration confirms the presence of a **flat optimum region** rather than a sharp peak. Importantly, the optimal parameters found by the automated search lie in the same neighborhood previously identified manually, thereby **reconfirming the hypotheses of the hard-coded analysis**.

From a modeling perspective, **C** controls the regularization of the linear classifier applied to ambiguous samples, **min_rule_purity** defines the confidence threshold for deterministic rule application, and **min_rule_support** regulates the statistical reliability of such rules. Together, these parameters define a precision–coverage trade-off that is shown to be stable across both manual and automated exploration.




In [1]:
import pandas as pd
import numpy as np
import re
import optuna
from collections import Counter, defaultdict

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

c:\Users\msist\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
DEV_PATH = "../../data/processed/development_processed.csv"

NUM_COLS = [
	"n_tokens",
	"title_len",
	"article_len",
	"title_ratio"
]

WORD_NG_MAX = 2
CHAR_NG_MAX = 5
MIN_DF      = 2
MAX_DF      = 0.9

RANDOM_STATE = 42


In [12]:
#Load data 

df = pd.read_csv(DEV_PATH)

X = df[["source", "text"] + NUM_COLS]
y = df["label"].values

X_tr, X_va, y_tr, y_va = train_test_split(
	X, y,
	test_size=0.2,
	stratify=y,
	random_state=RANDOM_STATE
)

In [13]:
# Rules
def tokenize_for_rules(text):
	if not isinstance(text, str):
		return []
	return re.findall(r"[a-z0-9_:/\.]+", text.lower())


def mine_pure_rules(texts, labels, min_support, min_purity):
	counts = defaultdict(lambda: Counter())

	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(y)] += 1

	rules = {}
	meta = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < min_support:
			continue

		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= min_purity:
			rules[tok] = best_class
			meta[tok] = (purity, total)

	return rules, meta
def apply_rules(texts, rules, meta):
	pred = np.full(len(texts), -1)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rules]

		if not hits:
			continue

		hits.sort(
			key=lambda t: (meta[t][0], meta[t][1]),
			reverse=True
		)

		pred[i] = rules[hits[0]]

	return pred


In [14]:
def make_model(C):
	pre = ColumnTransformer(
		[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, WORD_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=250_000
			), "text"),
			("c", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, CHAR_NG_MAX),
				min_df=MIN_DF,
				max_df=MAX_DF,
				sublinear_tf=True,
				max_features=300_000
			), "text"),
			("num", StandardScaler(), NUM_COLS)
		],
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([("pre", pre), ("clf", clf)])


In [15]:
#Optuna objective
def objective(trial):
	C = trial.suggest_float("C", 0.5, 2.0, log=True)

	min_rule_purity = trial.suggest_categorical(
		"min_rule_purity", [0.90, 0.93, 0.95]
	)

	min_rule_support = trial.suggest_categorical(
		"min_rule_support", [20, 30, 40]
	)

	# Stage 2
	model = make_model(C)
	model.fit(X_tr, y_tr)
	ml_pred = model.predict(X_va)

	# Stage 1
	rules, meta = mine_pure_rules(
		df.loc[X_tr.index, "text"],
		y_tr,
		min_rule_support,
		min_rule_purity
	)

	rule_pred = apply_rules(
		df.loc[X_va.index, "text"],
		rules,
		meta
	)

	final_pred = ml_pred.copy()
	mask = rule_pred != -1
	final_pred[mask] = rule_pred[mask]

	f1 = f1_score(y_va, final_pred, average="macro")

	trial.set_user_attr("rule_coverage", float(mask.mean()))

	return f1


In [16]:
#Run
study = optuna.create_study(
	direction="maximize",
	study_name="two_stage_regex",
	storage="sqlite:///two_stage_regex.db",
	load_if_exists=True
)

study.optimize(objective, n_trials=50)


[I 2026-01-17 00:55:31,666] Using an existing study with name 'two_stage_regex' instead of creating a new one.


[I 2026-01-17 01:03:13,014] Trial 2 finished with value: 0.7199678453016031 and parameters: {'C': 1.1315528950465592, 'min_rule_purity': 0.93, 'min_rule_support': 40}. Best is trial 2 with value: 0.7199678453016031.
[I 2026-01-17 01:09:49,171] Trial 3 finished with value: 0.720972448900483 and parameters: {'C': 0.6146165325747928, 'min_rule_purity': 0.93, 'min_rule_support': 20}. Best is trial 3 with value: 0.720972448900483.
[I 2026-01-17 01:16:27,116] Trial 4 finished with value: 0.7199277303757075 and parameters: {'C': 0.588283848886016, 'min_rule_purity': 0.9, 'min_rule_support': 40}. Best is trial 3 with value: 0.720972448900483.
[I 2026-01-17 01:23:52,176] Trial 5 finished with value: 0.7170233438668312 and parameters: {'C': 1.6618342257928216, 'min_rule_purity': 0.9, 'min_rule_support': 20}. Best is trial 3 with value: 0.720972448900483.
[I 2026-01-17 01:31:37,021] Trial 6 finished with value: 0.7194196055496888 and parameters: {'C': 1.5272223634029745, 'min_rule_purity': 0.93, 

In [17]:
print("Best F1:", study.best_value)
print("Best params:", study.best_params)

df_trials = study.trials_dataframe()
df_trials.to_csv("optuna_two_stage_results.csv", index=False)
print("Saved optuna_two_stage_results.csv")

Best F1: 0.7220136312451597
Best params: {'C': 0.9654966943081449, 'min_rule_purity': 0.93, 'min_rule_support': 30}
Saved optuna_two_stage_results.csv
